In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

### DIRECTORY AND FILE PATH CONFIGURATION

In [2]:
ROOT_DIR=Path.cwd().parent
DATASET_DIR= ROOT_DIR / "dataset"
CSV_FILE_PATH = DATASET_DIR / "raw_loan_dataset.csv"
print(ROOT_DIR)
print(DATASET_DIR)
print(CSV_FILE_PATH)

/home/mdev/Documents/ml/ds-ml-bootcamp
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset/raw_loan_dataset.csv


### DATASET (EDA)

In [3]:
df = pd.read_csv(CSV_FILE_PATH)
df = df.rename(columns={'Approved':'Status'})
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [4]:
rows,columns = df.shape
print("ROWS:",rows)
print("COLUMNS:",columns)

ROWS: 103
COLUMNS: 7


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB


In [6]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [7]:
def display_unique_values(col:str):
    print(f"\n{col}:",df[col].unique())

In [8]:
categorical_cols = ['Status','HasCollateral','PreviousDefaults']
for col in categorical_cols:
    display_unique_values(col)


Status: ['No' 'Yes' 'Approved' 'Rejected' 'approved' 'rejected' 'YES' 'NO']

HasCollateral: ['Yes' 'Y' 'No' 'N' nan 'yse' 'yes']

PreviousDefaults: ['No' nan 'Yes' '1' '0' 'Y' 'N']


### CLEANING AND FORMATTING

In [9]:
def clean_numeric_col(col:str):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(r'[^0-9.]','',regex=True).astype(str)
    df[col] = df[col].str.strip()
    df[col] = df[col].replace('',np.nan)
    df[col] = pd.to_numeric(df[col],errors="coerce")

In [10]:
numeric_cols = ['Income','LoanAmount','CreditScore']
for col in numeric_cols:
    clean_numeric_col(col)
    print(df[col])


Cleaning column: Income
0      108810.0
1       96482.0
2       28478.0
3       25851.0
4       38396.0
         ...   
98      97529.0
99      62268.0
100    108810.0
101     96482.0
102     28478.0
Name: Income, Length: 103, dtype: float64

Cleaning column: LoanAmount
0      25800.0
1      11200.0
2      12100.0
3       7000.0
4      10700.0
        ...   
98     15900.0
99         NaN
100    25800.0
101    11200.0
102    12100.0
Name: LoanAmount, Length: 103, dtype: float64

Cleaning column: CreditScore
0      537.0
1      524.0
2        NaN
3      561.0
4      527.0
       ...  
98     629.0
99     720.0
100    537.0
101    524.0
102      NaN
Name: CreditScore, Length: 103, dtype: float64


In [11]:
categorical_map = {
    '1':'Yes','Approved':'Yes','approved':'Yes','Y':'Yes','yse':'Yes',
    'Rejected':'No','rejected':'No','N':'No','0':'No'
    
}

def clean_format_categorical_col(col:str,categorical_map:dict):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(categorical_map).astype(str)
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].replace({'Nan':np.nan})

In [12]:
for col in categorical_cols:
    clean_format_categorical_col(col,categorical_map)
    display_unique_values(col)


Cleaning column: Status

Status: ['No' 'Yes']

Cleaning column: HasCollateral

HasCollateral: ['Yes' 'No' nan]

Cleaning column: PreviousDefaults

PreviousDefaults: ['No' nan 'Yes']


In [13]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

### FILL MISSING VALUES

In [14]:
def fill_with_median(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].median())
        
def fill_with_mode(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].mode()[0])

In [15]:
for col in numeric_cols + ['EmploymentYears']:
    fill_with_median(col)
df.isna().sum()


Filling column: Income

Filling column: LoanAmount

Filling column: CreditScore

Filling column: EmploymentYears


Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [16]:
for col in categorical_cols:
    fill_with_mode(col)
    display_unique_values(col)
df.isna().sum()


Filling column: Status

Status: ['No' 'Yes']

Filling column: HasCollateral

HasCollateral: ['Yes' 'No']

Filling column: PreviousDefaults

PreviousDefaults: ['No' 'Yes']


Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Status              0
dtype: int64

### REMOVE DUPLICATES

In [17]:
before = df.shape
df = df.drop_duplicates()
print(f"Before dropping {before}(row,col) after dropped {df.shape}(row,col)")

Before dropping (103, 7)(row,col) after dropped (100, 7)(row,col)
